# Convert the fine-tuned Qwen2.5-7B to one GGUF file for the laptop

Output: `qwen7b-mcq-Q4_K_M.gguf` (about 4.7 GB), which runs on a CPU-only laptop with llama.cpp or Ollama.

**Kaggle settings:** Accelerator **None (CPU)**, so there is no GPU queue. Internet ON.
Add Input → Your Work → the fine-tuning notebook (for `final_adapter`, or `qwen7b_mcq_adapter.zip`).
Do not use free Colab: merging needs about 16 GB of RAM, and Colab has 12 GB. Kaggle CPU sessions have about 30 GB.

Steps: merge the LoRA adapter into the full 16-bit model → convert to GGUF → shrink it to 4-bit (Q4_K_M) → quick test.
Only temporary copies inside this Kaggle session are removed, and only if the disk gets full. Your adapter and outputs are never touched.
Time: about 45 to 75 minutes (estimate).

In [ ]:
!pip install -q -U "transformers>=4.56" peft accelerate sentencepiece safetensors
# Kaggle ships torchao 0.10, which new peft rejects. It is not needed for a CPU merge, so remove it from this session only.
!pip uninstall -y -q torchao

In [ ]:
import os, glob, json, shutil, subprocess, time, zipfile
os.environ["HF_HOME"] = "/tmp/hf"                 # big temporary downloads go to /tmp, not to the saved output
WORK, TMP = "/kaggle/working", "/tmp"
def disk(p="/tmp"): return f"{shutil.disk_usage(p).free/1e9:.0f} GB free on {p}"
def ram():
    return f"{os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/1e9:.0f} GB RAM"
print(ram(), "|", disk("/tmp"), "|", disk(WORK))

def find_files(root, name):
    out = []
    for d, _, files in os.walk(root, followlinks=True):
        if name in files: out.append(os.path.join(d, name))
    return out
hits = [h for h in find_files("/kaggle/input", "adapter_config.json") if "final_adapter" in h]        or [h for h in find_files("/kaggle/input", "adapter_config.json") if "checkpoint" not in h]        or sorted(find_files("/kaggle/input", "adapter_config.json"))[-1:]   # Kaggle unzips uploaded zips, so files can sit at the dataset root
if not hits:
    for z in find_files("/kaggle/input", "qwen7b_mcq_adapter.zip"):
        zipfile.ZipFile(z).extractall(f"{TMP}/final_adapter")
    hits = find_files(f"{TMP}/final_adapter", "adapter_config.json")
if not hits:
    print("Inputs this notebook can see:")
    for d, dirs, files in os.walk("/kaggle/input", followlinks=True):
        if d.count("/") <= 6: print("  ", d, files[:5])
    raise SystemExit("Fine-tuned model not found. Add Input -> Your Work -> Notebooks -> your fine-tuning notebook, "
                     "check it appears under Input in the right panel, then Save Version again.")
ADAPTER_DIR = os.path.dirname(hits[0])
cfg = json.load(open(hits[0]))
if not os.path.exists(f"{ADAPTER_DIR}/train_config.json"):      # checkpoint folders have no train_config.json
    tc = find_files("/kaggle/input", "train_config.json")
    if tc: shutil.copytree(ADAPTER_DIR, f"{TMP}/adapter", dirs_exist_ok=True); shutil.copy(tc[0], f"{TMP}/adapter/"); ADAPTER_DIR = f"{TMP}/adapter"
print("adapter:", ADAPTER_DIR, "| trained on:", cfg.get("base_model_name_or_path"), "| r =", cfg.get("r"))

## 1. Merge the adapter into the full 16-bit Qwen2.5-7B-Instruct (CPU, about 15 to 25 min)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import sys, peft.tuners.lora.torchao, peft.utils.quantization_utils
# Kaggle's torchao 0.10 is too old for new peft, and a CPU merge never uses torchao, so tell peft it is not installed
for name, mod in list(sys.modules.items()):
    if name.startswith("peft") and hasattr(mod, "is_torchao_available"):
        mod.is_torchao_available = lambda: False
BASE = "Qwen/Qwen2.5-7B-Instruct"                 # same weights the 4-bit training copy was made from
MERGED = f"{TMP}/qwen7b-mcq-merged"
t0 = time.time()
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, low_cpu_mem_usage=True, device_map={"": "cpu"})
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model = model.merge_and_unload()
model.save_pretrained(MERGED, safe_serialization=True, max_shard_size="2GB")
AutoTokenizer.from_pretrained(BASE).save_pretrained(MERGED)   # original Qwen tokenizer + chat template
del model, base; import gc; gc.collect()
print(f"merged in {(time.time()-t0)/60:.1f} min |", disk())
if shutil.disk_usage(TMP).free < 25e9:            # free the temporary base download only if space is short
    shutil.rmtree("/tmp/hf", ignore_errors=True); print("removed temporary base download |", disk())

## 2. Build llama.cpp (CPU, about 10 to 15 min)

In [ ]:
!cd /tmp && rm -rf llama.cpp && git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!cd /tmp/llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build build --config Release -j 4 > /tmp/build.log 2>&1; tail -n 3 /tmp/build.log
!pip install -q /tmp/llama.cpp/gguf-py protobuf
!ls /tmp/llama.cpp/build/bin | head -50

## 3. Convert to GGUF and shrink to 4-bit Q4_K_M

In [ ]:
F16 = f"{TMP}/qwen7b-mcq-f16.gguf"
OUT = f"{WORK}/qwen7b-mcq-Q4_K_M.gguf"
t0 = time.time()
r = subprocess.run(["python", "/tmp/llama.cpp/convert_hf_to_gguf.py", MERGED, "--outtype", "f16", "--outfile", F16],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:]); assert r.returncode == 0 and os.path.exists(F16), "conversion failed (see above)"
r = subprocess.run(["/tmp/llama.cpp/build/bin/llama-quantize", F16, OUT, "Q4_K_M"], capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:]); assert r.returncode == 0 and os.path.exists(OUT), "quantize failed (see above)"
print(f"done in {(time.time()-t0)/60:.1f} min ->", OUT, f"{os.path.getsize(OUT)/1e9:.2f} GB")

## 4. Quick test: one MCQ from the GGUF file on CPU (about 2 to 5 min)

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MERGED)
SYSTEM = ("You are a B.Tech exam setter. From the context, write ONE multiple-choice question of the requested difficulty. "
          "Exactly one option is correct; the three wrong options must be different from each other, plausible and clearly wrong. "
          "Check any calculation before answering. Reply in exactly this format: question: <question> options: (A) <option> "
          "(B) <option> (C) <option> (D) <option> answer: <letter> explanation: <one or two sentences>")   # same as training
if os.path.exists(f"{ADAPTER_DIR}/train_config.json"):
    SYSTEM = json.load(open(f"{ADAPTER_DIR}/train_config.json")).get("system", SYSTEM)
user = ("generate mcq | branch: CSE | subject: Operating Systems | topic: Deadlock | difficulty: medium | context: A deadlock is a situation where "
        "processes are blocked because each holds a resource and waits for another. Four conditions must hold: mutual exclusion, "
        "hold and wait, no preemption and circular wait. The Banker's algorithm avoids deadlock by granting a request only if the "
        "system stays in a safe state.")
prompt = tok.apply_chat_template([{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
                                 tokenize=False, add_generation_prompt=True)
open("/tmp/prompt.txt", "w").write(prompt)
binp = "/tmp/llama.cpp/build/bin/"
cli = [binp + "llama-completion" if os.path.exists(binp + "llama-completion") else binp + "llama-cli", "-no-cnv"]   # -no-cnv: plain one-shot run, no chat mode waiting for typing
t0 = time.time()
r = subprocess.run(cli + ["-m", OUT, "-f", "/tmp/prompt.txt", "-n", "300", "--temp", "0.7", "-t", "4", "--no-display-prompt"],
                   capture_output=True, text=True, timeout=1800, stdin=subprocess.DEVNULL)
print(r.stdout[-2500:] or r.stderr[-2500:])
print(f"\n{(time.time()-t0):.0f} s on Kaggle's 4 CPU cores (your laptop has 10 cores, so similar or faster)")

## 5. Save it
Download `qwen7b-mcq-Q4_K_M.gguf` from the Output tab, and also click **Output → New Dataset** to keep a copy on Kaggle.